# F1 Race Analytics  Modeling II: Linear Regression
**Question:** Can we predict lap time (continuous target) from tire age, lap number, and compound?  
**CRISP-DM Phase:** Modeling → Evaluation  
**Method:** Linear Regression (OLS)  predicts a number

In [ ]:
%matplotlib inline
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# reload same session from notebook 01
fastf1.Cache.enable_cache('cache')
session = fastf1.get_session(2024, 'Japanese Grand Prix', 'R')
session.load()

laps = session.laps
clean_laps = laps.pick_accurate()

ver_laps = clean_laps.pick_drivers('VER').copy()
pia_laps = clean_laps.pick_drivers('PIA').copy()

ver_laps['LapTimeSec'] = ver_laps['LapTime'].dt.total_seconds()
pia_laps['LapTimeSec'] = pia_laps['LapTime'].dt.total_seconds()

print(f"VER: {len(ver_laps)} laps | PIA: {len(pia_laps)} laps")

## Features & Target

| Role | Variable | Description |
|------|----------|-------------|
| Feature X₁ | `TyreLife` | Laps on current tire set |
| Feature X₂ | `LapNumber` | Race lap number |
| Feature X₃ | `Compound_enc` | Tire compound (MEDIUM=0, HARD=1) |
| **Target y** | `LapTimeSec` | Lap time in seconds |

Not standardizing features  keeping coefficients in real units (seconds per lap) for interpretability.

In [ ]:
def prepare_features(driver_laps):
    df = driver_laps.copy()
    # encode compound as numeric: MEDIUM=0, HARD=1
    df['Compound_enc'] = df['Compound'].map({'MEDIUM': 0, 'HARD': 1})
    df = df.dropna(subset=['LapTimeSec', 'TyreLife', 'LapNumber', 'Compound_enc'])
    X = df[['TyreLife', 'LapNumber', 'Compound_enc']]  # features
    y = df['LapTimeSec']                                # target
    return X, y, df

X_ver, y_ver, ver_df = prepare_features(ver_laps)
X_pia, y_pia, pia_df = prepare_features(pia_laps)

print(X_ver.head())
print(f"\nVER: {X_ver.shape} | PIA: {X_pia.shape}")

## Train / Test Split  80/20
Train the model on 80% of laps, evaluate on the remaining 20% (unseen data).

In [ ]:
# random_state=1907 for reproducibility
X_ver_tr, X_ver_te, y_ver_tr, y_ver_te = train_test_split(X_ver, y_ver, test_size=0.2, random_state=1907)
X_pia_tr, X_pia_te, y_pia_tr, y_pia_te = train_test_split(X_pia, y_pia, test_size=0.2, random_state=1907)

print(f"VER train/test: {len(X_ver_tr)}/{len(X_ver_te)}")
print(f"PIA train/test: {len(X_pia_tr)}/{len(X_pia_te)}")

## Model Training  OLS
LinearRegression uses **Ordinary Least Squares (OLS)**: minimizes `Σ(yᵢ - ŷᵢ)²` to find the best-fit coefficients.

In [ ]:
model_ver = LinearRegression()
model_ver.fit(X_ver_tr, y_ver_tr)  # learn coefficients β₀, β₁, β₂, β₃

model_pia = LinearRegression()
model_pia.fit(X_pia_tr, y_pia_tr)

y_ver_pred = model_ver.predict(X_ver_te)
y_pia_pred = model_pia.predict(X_pia_te)

## Evaluation  MSE
**MSE** (Mean Squared Error) = average squared difference between real and predicted values. Lower = better.  
**RMSE** = √MSE  same units as lap time (seconds), easier to interpret.  
**R²**  proportion of variance explained by the model (1.0 = perfect).

In [ ]:
def evaluate(y_real, y_pred, name):
    mse = mean_squared_error(y_real, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_real, y_pred)
    print(f"{name}: MSE={mse:.3f} | RMSE={rmse:.3f}s | R²={r2:.3f}")
    return rmse, r2

rmse_ver, r2_ver = evaluate(y_ver_te, y_ver_pred, 'VER')
rmse_pia, r2_pia = evaluate(y_pia_te, y_pia_pred, 'PIA')

## Coefficients (β)
Model formula: `ŷ = β₀ + β₁·TyreLife + β₂·LapNumber + β₃·Compound_enc`

Each coefficient shows the effect of one feature on predicted lap time, holding all others constant.

In [ ]:
features = ['TyreLife (β₁)', 'LapNumber (β₂)', 'Compound Hard vs Med (β₃)']
for name, model in [('VER', model_ver), ('PIA', model_pia)]:
    print(f"\n{name}  intercept β₀ = {model.intercept_:.2f}s")
    for f, c in zip(features, model.coef_):
        print(f"  {f:<35} {c:+.4f}s per unit")

## Chart 1  Actual vs Predicted
Points close to the diagonal = accurate predictions. Scatter around the line = residual error.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_real, y_pred, name, color in zip(
    axes,
    [y_ver_te, y_pia_te],
    [y_ver_pred, y_pia_pred],
    ['Verstappen', 'Piastri'],
    ['#0600EF', '#FF8000']
):
    ax.scatter(y_real, y_pred, color=color, alpha=0.7, s=50)
    lim = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
    ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Lap Time (s)')
    ax.set_ylabel('Predicted Lap Time (s)')
    ax.set_title(f'{name}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Actual vs Predicted Lap Times', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/05_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 2  Tire Degradation (Feature X₁ vs Target y)
Shows how TyreLife affects LapTimeSec per compound. Trend line = linear regression fit per compound.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
compound_colors = {'MEDIUM': '#FFD700', 'HARD': '#AAAAAA'}

for ax, df, name in zip(axes, [ver_df, pia_df], ['Verstappen', 'Piastri']):
    for compound in df['Compound'].unique():
        sub = df[df['Compound'] == compound]
        c = compound_colors.get(compound, 'grey')
        ax.scatter(sub['TyreLife'], sub['LapTimeSec'], color=c, label=compound, alpha=0.8, s=30)
        if len(sub) > 2:
            z = np.polyfit(sub['TyreLife'], sub['LapTimeSec'], 1)
            x_line = np.linspace(sub['TyreLife'].min(), sub['TyreLife'].max(), 50)
            ax.plot(x_line, np.poly1d(z)(x_line), color=c, linewidth=2, linestyle='--')

    ax.set_xlabel('Tire Age  TyreLife (laps)')
    ax.set_ylabel('Lap Time  LapTimeSec (s)')
    ax.set_title(f'{name}  Tire Degradation')
    ax.legend(title='Compound')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/06_tire_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

## Chart 3  Residuals
**Residual = actual − predicted** (yᵢ − ŷᵢ). Random scatter around zero = model is unbiased. Patterns = something the model missed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, y_real, y_pred, name, color in zip(
    axes,
    [y_ver_te, y_pia_te],
    [y_ver_pred, y_pia_pred],
    ['Verstappen', 'Piastri'],
    ['#0600EF', '#FF8000']
):
    residuals = y_real.values - y_pred  # yᵢ - ŷᵢ
    ax.scatter(y_pred, residuals, color=color, alpha=0.7, s=50)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted Lap Time (s)')
    ax.set_ylabel('Residual (s)')
    ax.set_title(f'{name}  Residuals')
    ax.grid(True, alpha=0.3)

plt.suptitle('Residual Plot', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/07_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("=== Model Results ===")
print(f"VER: RMSE={rmse_ver:.3f}s  R²={r2_ver:.3f}")
print(f"PIA: RMSE={rmse_pia:.3f}s  R²={r2_pia:.3f}")
print(f"\nTire degradation coefficient β₁ (s per lap on tire):")
print(f"  VER: {model_ver.coef_[0]:+.4f}s")
print(f"  PIA: {model_pia.coef_[0]:+.4f}s")